# Preprocesamiento de Variables
## Modelo de Riesgo Crediticio — Scotiabank Perú

**Objetivo:** Aplicar y documentar cada decisión de preprocesamiento con justificación
técnica y de negocio. El pipeline debe ser serializable para garantizar que producción
aplique exactamente las mismas transformaciones que entrenamiento.

**Estructura del notebook:**
1. Carga de datos
2. Análisis de valores centinela (-9999998): ¿son nulos o condiciones de negocio?
3. Estrategia de flags: separar centinelas de nulos reales
4. Outliers: detección y capping
5. Imputación de nulos reales
6. Encoding de variable categórica X9
7. Pipeline completo y validación
8. Guardado de artefactos
---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sys.path.insert(0, str(Path('..').resolve()))
from src.config import (
    RAW_DATA_FILE, PROCESSED_DATA_DIR, MODELS_DIR,
    NUMERIC_FEATURES, TARGET_COL, ID_COL, BASE_COL, SENTINEL_VALUE
)
from src.data.preprocessing import (
    load_raw_data, replace_sentinels, missing_summary, sentinel_summary,
    detect_outliers_iqr, RiskPreprocessor, prepare_train_test,
    SENTINEL_COLS, INFORMATIVE_NULL_COLS
)
import joblib

warnings.filterwarnings('ignore')
PALETTE = ['#E31837', '#002D72', '#00A3E0', '#6D6E71']
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print('Setup completo')

## 1. Carga de Datos

In [ ]:
df_raw = load_raw_data(RAW_DATA_FILE)
print(f'Dataset: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
print(f'TRAIN: {(df_raw[BASE_COL]=="TRAIN").sum():,} | TEST: {(df_raw[BASE_COL]=="TEST").sum():,}')
display(df_raw.head(3))

## 2. Análisis de Valores Centinela (-9999998)

### La pregunta clave en riesgo: ¿este -9999998 es un nulo técnico o una condición de negocio?

En sistemas bancarios legados (Temenos T24, Oracle FLEXCUBE), el valor -9999998 se usa
cuando el campo numérico no admite NULL. Hay dos interpretaciones posibles:

- **Interpretacion A (nulo tecnico):** El dato existe pero no pudo registrarse → imputar.
- **Interpretacion B (condicion de negocio):** El cliente NO TIENE el producto/historial
  asociado a esa variable → es un SEGMENTO distinto, no un dato faltante.

La diferencia tiene impacto directo en el modelo: en el caso B, el -9999998 lleva
informacion predictiva propia que se pierde si simplemente se imputa con la mediana.

In [ ]:
# Cuantos registros tienen centinela y tienen nulos reales?
print('=== Presencia de centinelas por columna ===')
display(sentinel_summary(df_raw))

print('\n=== Nulos reales (celdas vacias) en X3 y X4 ===')
for col in SENTINEL_COLS:
    n_real_null = df_raw.loc[df_raw[col] != SENTINEL_VALUE, col].isna().sum()
    print(f'  {col}: {n_real_null} nulos reales (no centinela)')

print('\nX3 y X4 solo tienen centinelas — NINGUN nulo real.')
print('El -9999998 es la UNICA fuente de ausencia en estas columnas.')

In [ ]:
# Verificar que los valores reales de X3 y X4 son SIEMPRE positivos
# (nunca 0, nunca negativos) — confirma que son saldos/montos de un producto
for col in SENTINEL_COLS:
    real = df_raw.loc[df_raw[col] != SENTINEL_VALUE, col].dropna()
    print(f'{col} (valores reales): min={real.min():,.0f}  max={real.max():,.0f}  '
          f'pct_positivos={(real > 0).mean()*100:.1f}%  pct_cero={(real == 0).mean()*100:.1f}%')

print('\nConclucion: X3 y X4 son variables de saldo/monto (siempre > 0 cuando existen).')
print('Un centinela indica que el cliente NO TIENE ese producto financiero.')

In [ ]:
# PRUEBA CRITICA: la distribucion de TARGET difiere entre clientes con centinela y sin el?
# Si la diferencia es significativa → el centinela es informativo del riesgo

print('=== Impacto del centinela sobre TARGET ===')
print(f'  {"Variable":<6} {"N centinela":>12} {"TARGET medio (centinela)":>25} '
      f'{"TARGET medio (real)":>22} {"Diferencia":>12} {"KS p-valor":>12}')
print('  ' + '-' * 95)

for col in SENTINEL_COLS:
    mask_s = df_raw[col] == SENTINEL_VALUE
    mask_r = ~mask_s & df_raw[col].notna()
    t_s = df_raw.loc[mask_s, TARGET_COL]
    t_r = df_raw.loc[mask_r, TARGET_COL]
    ks_stat, ks_p = stats.ks_2samp(t_s, t_r)
    diff_pct = (t_s.mean() - t_r.mean()) / t_r.mean() * 100
    print(f'  {col:<6} {mask_s.sum():>12,} {t_s.mean():>25,.0f} '
          f'{t_r.mean():>22,.0f} {diff_pct:>11.1f}% {ks_p:>12.2e}')

print('\nAmbas distribuciones son SIGNIFICATIVAMENTE DISTINTAS (p aprox 0).')
print('El centinela NO es un nulo aleatorio: IDENTIFICA UN SEGMENTO DE MENOR INGRESO/RIESGO.')

In [ ]:
# Analisis por combinacion: 4 segmentos de negocio claramente distintos
mask_x3 = df_raw['X3'] == SENTINEL_VALUE
mask_x4 = df_raw['X4'] == SENTINEL_VALUE

segmentos = {
    'Sin X3 y sin X4  (cliente sin ambos productos)': mask_x3 & mask_x4,
    'Sin X3, con X4   (cliente sin producto X3)    ': mask_x3 & ~mask_x4,
    'Con X3, sin X4   (cliente sin producto X4)    ': ~mask_x3 & mask_x4,
    'Con X3 y con X4  (cliente con ambos productos)': ~mask_x3 & ~mask_x4,
}

print('Segmentos de negocio por patron de centinela X3/X4:')
print(f'  {"Segmento":<50} {"N":>7}  {"Pct":>6}  {"Mediana TARGET":>15}')
print('  ' + '-' * 82)
for label, mask in segmentos.items():
    n = mask.sum()
    med = df_raw.loc[mask, TARGET_COL].median()
    print(f'  {label:<50} {n:>7,}  {n/len(df_raw)*100:>5.1f}%  {med:>15,.0f}')

print('\nExisten 4 segmentos con TARGET mediana distintas (5,512 → 6,586 → 6,773 → 7,656).')
print('Tratarlos igual con imputacion de mediana global BORRARIA esta distincion de negocio.')

In [ ]:
# Visualizar: distribucion de TARGET por grupo centinela
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, col in enumerate(SENTINEL_COLS):
    mask_s = df_raw[col] == SENTINEL_VALUE
    axes[i].hist(df_raw.loc[~mask_s, TARGET_COL], bins=60, alpha=0.6,
                 color=PALETTE[1], label=f'Con {col} (valor real)', density=True)
    axes[i].hist(df_raw.loc[mask_s, TARGET_COL], bins=60, alpha=0.6,
                 color=PALETTE[0], label=f'Sin {col} (centinela)', density=True)
    axes[i].set_title(f'TARGET segun {col}', fontweight='bold')
    axes[i].set_xlabel('TARGET')
    axes[i].set_ylabel('Densidad')
    axes[i].legend()

plt.suptitle('Distribucion de TARGET: clientes con/sin centinela en X3 y X4',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_sentinel_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Estrategia de Flags: Separar Centinelas de Nulos Reales

### Conclusion del analisis

El -9999998 en X3 y X4 es una **condicion de negocio** (cliente sin producto), no un
nulo tecnico. Esto cambia la estrategia de tratamiento:

| Tipo de ausencia | Variables | Significado | Flag a crear | Nombre |
|-----------------|-----------|-------------|-------------|--------|
| Centinela (-9999998) | X3, X4 | Cliente sin producto financiero | Antes de reemplazar | `X3_sin_registro`, `X4_sin_registro` |
| Nulo real (celda vacia) | X1, X2, X5, X7 | Dato no disponible | Antes de imputar | `X1_sin_dato`, `X2_sin_dato`, `X5_sin_dato`, `X7_sin_dato` |

### Por que el orden importa

Los flags de centinela deben crearse **ANTES** de `replace_sentinels()`, ya que una vez
convertidos a NaN no se puede distinguir si el NaN vino de un centinela o de otro origen.

### Por que la imputacion con mediana global es insuficiente por si sola

Para X3: la mediana de clientes CON producto es ~7,500. Si imputamos con este valor
a los 19,922 clientes SIN producto (que tienen TARGET mediana ~5,512), el modelo
vera un X3 'normal' y no podra distinguirlos. El flag `X3_sin_registro` resuelve esto:
el modelo puede aprender que X3_sin_registro=1 predice TARGET menor.

In [ ]:
# Analisis de nulos reales: tambien son informativos del riesgo?
print('=== Impacto de nulos reales sobre TARGET ===')
print(f'  {"Variable":<6} {"N nulos":>10} {"TARGET medio (nulo)":>22} '
      f'{"TARGET medio (real)":>22} {"Diferencia":>12}')
print('  ' + '-' * 75)

for col in INFORMATIVE_NULL_COLS:
    mask_null = df_raw[col].isna()
    mask_real = ~mask_null & (df_raw[col] != SENTINEL_VALUE)
    if mask_null.sum() == 0:
        continue
    t_null = df_raw.loc[mask_null, TARGET_COL]
    t_real = df_raw.loc[mask_real, TARGET_COL]
    diff_pct = (t_null.mean() - t_real.mean()) / t_real.mean() * 100
    print(f'  {col:<6} {mask_null.sum():>10,} {t_null.mean():>22,.0f} '
          f'{t_real.mean():>22,.0f} {diff_pct:>11.1f}%')

print('\nTodos los nulos reales son informativos (diferencia de medias > 4%).')
print('Los flags de disponibilidad de dato son necesarios para capturar esta informacion.')

## 4. Deteccion y Tratamiento de Outliers

**Decision: Capping (Winsorisacion) con IQR factor=3**

**Alternativas evaluadas:**
- *Eliminacion de filas:* Los clientes atipicos pueden ser los de mayor riesgo — perdida de informacion inaceptable.
- *Capping percentil 1-99:* Mas agresivo, corta variabilidad legitima en ingresos altos.
- *Capping IQR×3 (elegida):* Solo afecta outliers extremos (> 3 IQRs del cuartil). Estandar en scoring bancario.

**Anti-leakage:** Los bounds se aprenden SOLO en TRAIN y se aplican identicamente en TEST y produccion.

In [ ]:
df = replace_sentinels(df_raw.copy())
train_df = df[df[BASE_COL] == 'TRAIN'].copy()

x1_original = train_df['X1'].dropna()
Q1, Q3 = x1_original.quantile(0.25), x1_original.quantile(0.75)
IQR = Q3 - Q1
x1_capped = x1_original.clip(Q1 - 3*IQR, Q3 + 3*IQR)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(x1_original, bins=80, color=PALETTE[0], edgecolor='white', alpha=0.8)
axes[0].set_title('X1: Distribucion original (con outliers)', fontweight='bold')
axes[1].hist(x1_capped, bins=80, color=PALETTE[1], edgecolor='white', alpha=0.8)
axes[1].set_title('X1: Distribucion tras capping IQR×3', fontweight='bold')

n_affected = (x1_original != x1_capped).sum()
plt.suptitle(f'Impacto del Capping en X1 | Afectados: {n_affected:,} ({n_affected/len(x1_original)*100:.1f}%)',
             fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/02_outlier_capping.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Imputacion de Nulos Reales

**Estrategia: Mediana de cada columna calculada SOLO sobre valores reales en TRAIN**

**Por que mediana y no media:**
Variables financieras (ingresos, saldos) son right-skewed. La mediana es mas robusta.

**Por que la mediana NO contamina los centinelas de X3/X4:**
El SimpleImputer se ajusta DESPUES de replace_sentinels → la mediana de X3 se calcula
solo sobre el 60% de clientes que SI tienen el producto. La imputacion llena el NaN
con esa mediana, pero el flag `X3_sin_registro` ya captura al grupo sin producto,
permitiendo al modelo aprender el efecto diferencial.

In [ ]:
# Verificar que medianas de TRAIN y TEST son similares (imputacion valida cross-split)
comparison = []
cols_with_missing = ['X1', 'X2', 'X3', 'X4', 'X5', 'X7', 'X11', 'X12']

for col in cols_with_missing:
    train_med = df[df[BASE_COL] == 'TRAIN'][col].median()
    test_med  = df[df[BASE_COL] == 'TEST'][col].median()
    diff_pct  = abs(train_med - test_med) / (abs(train_med) + 1e-6) * 100
    comparison.append({'variable': col,
                       'mediana_train': round(train_med, 2),
                       'mediana_test':  round(test_med, 2),
                       'diferencia_%':  round(diff_pct, 2)})

display(pd.DataFrame(comparison))
print('Medianas de TRAIN y TEST son similares: imputacion con mediana de TRAIN es valida.')

## 6. Encoding de Variable Categorica X9 (Distrito)

**Estrategia: One-Hot Encoding top-25 distritos + OTROS**

- *Label Encoding descartado:* implica orden artificial entre distritos.
- *Target Encoding descartado:* riesgo de leakage si no se usa k-fold en entrenamiento.
- *OHE top-25 elegida:* captura los distritos con suficiente masa muestral (> 200 obs
  cada uno). El resto va a OTROS para evitar memorizar ruido.

**Nota sobre X9 nulo:** Los clientes sin distrito tienen TARGET +68% respecto al promedio.
Al asignarlos a la categoria 'DESCONOCIDO', el modelo puede aprender ese efecto positivo.

## 7. Pipeline Completo y Validacion

In [ ]:
# Ejecutar el pipeline completo (fit en TRAIN, transform en ambos splits)
X_train, y_train, X_test, y_test, preprocessor = prepare_train_test(df_raw)

print('=== Dimensiones tras preprocesamiento ===')
print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'\nFlags de condicion de negocio (centinela):')
print([c for c in X_train.columns if '_sin_registro' in c])
print(f'\nFlags de disponibilidad de dato (nulo real):')
print([c for c in X_train.columns if '_sin_dato' in c])
print(f'\nDummies de X9:')
print([c for c in X_train.columns if c.startswith('X9_')])

In [ ]:
# Verificar que los flags capturan exactamente los centinelas originales
train_raw = df_raw[df_raw[BASE_COL] == 'TRAIN'].reset_index(drop=True)

for col in ['X3', 'X4']:
    n_sentinel_orig = (train_raw[col] == SENTINEL_VALUE).sum()
    n_flag          = X_train[f'{col}_sin_registro'].sum()
    ok = n_sentinel_orig == n_flag
    print(f'{col}_sin_registro: {n_flag:,} flags | originales: {n_sentinel_orig:,} | Match: {ok}')

# Verificar ausencia de missing en numericas tras imputacion
num_cols = [c for c in X_train.columns if X_train[c].dtype in ['float64', 'int64']]
missing_train = X_train[num_cols].isnull().sum().sum()
missing_test  = X_test[num_cols].isnull().sum().sum()

assert missing_train == 0, 'Error: quedan NaN en X_train'
assert missing_test  == 0, 'Error: quedan NaN en X_test'
print(f'\nMissing en X_train (numericas): {missing_train}')
print(f'Missing en X_test  (numericas): {missing_test}')
print('Sin valores faltantes en features procesadas')

In [ ]:
# Verificar que no quedan centinelas en ningun valor numerico
sentinel_remaining = (X_train.values == SENTINEL_VALUE).sum()
assert sentinel_remaining == 0, 'Error: centinelas no reemplazados'
print(f'Centinelas (-9999998) restantes: {sentinel_remaining}')

In [ ]:
# Comparar distribuciones originales vs procesadas para las features clave
df_clean = replace_sentinels(df_raw.copy())
df_train_raw = df_clean[df_clean[BASE_COL] == 'TRAIN']

check_cols = ['X1', 'X2', 'X3', 'X4', 'X5', 'X7', 'X10', 'X11']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(check_cols):
    axes[i].hist(df_train_raw[col].dropna(), bins=40, alpha=0.5,
                 color=PALETTE[0], label='Original', density=True)
    if col in X_train.columns:
        axes[i].hist(X_train[col], bins=40, alpha=0.5,
                     color=PALETTE[1], label='Procesado', density=True)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].legend(fontsize=7)

fig.suptitle('Distribuciones: Original vs Procesado (TRAIN)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_preprocessing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Guardado de Artefactos

In [ ]:
train_processed = X_train.copy()
train_processed[TARGET_COL] = y_train.values

test_processed = X_test.copy()
test_processed[TARGET_COL] = y_test.values

train_processed.to_parquet(PROCESSED_DATA_DIR / 'train_processed.parquet', index=False)
test_processed.to_parquet(PROCESSED_DATA_DIR / 'test_processed.parquet', index=False)

# Serializar el preprocessor (mismo objeto que se usara en la API de produccion)
joblib.dump(preprocessor, MODELS_DIR / 'preprocessor.pkl')

print('Artefactos guardados:')
print(f'  data/processed/train_processed.parquet — {len(train_processed):,} filas')
print(f'  data/processed/test_processed.parquet  — {len(test_processed):,} filas')
print(f'  models/preprocessor.pkl')

## Resumen de Decisiones de Preprocesamiento

| Paso | Tecnica | Justificacion |
|------|---------|---------------|
| Centinelas X3/X4 | Flag `_sin_registro` ANTES de reemplazar | Condicion de negocio, no nulo tecnico. TARGET 6-18% menor en ese segmento. |
| Nulos reales | Flag `_sin_dato` ANTES de imputar | Dato no disponible; clientela distinta (TARGET 4-27% menor). |
| Imputacion | Mediana de valores reales en TRAIN | Robusta a skew; calcula sobre clientes CON producto (sin contaminar). |
| Outliers | Capping IQR×3 (bounds de TRAIN) | Preserva variabilidad legitima; anti-leakage. |
| X9 nulo | Categoria 'DESCONOCIDO' en OHE | Clientes sin distrito tienen TARGET +68% → categoria propia. |
| X9 | OHE top-25 distritos + OTROS | Sin leakage; top-25 cubre >85% de los datos. |
| Orden de pasos | Flags centinela → Replace → Flags nulos → Capping → Imputar → Encode | El orden garantiza que ningun flag sea contaminado por pasos anteriores. |

**El siguiente paso es `03_glm_model.ipynb`**